In [31]:
import pandas as pd
import numpy as np
import gamspy as gp
!gamspy install license 6378606e-e22d-413d-9851-2a2cfa5f6ac2
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Problem

Der Befehl "gamspy" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.


In [33]:
df = pd.read_excel("RVM_data_project (2).xlsx")

df = df.sample(n=50, random_state=42)
 
users = [str(n) for n in range(len(df))]
machines = [str(i) for i in range(1,36)]

In [35]:
B_DIST = -0.116
B_DIST_INDOOR = 0.123
B_INDOOR_AGE = -0.330
B_INDOOR_FEMALE = 0.368
B_BUILDING = 1.58
B_BUSINESS = -2.89
B_CAMPUS = -0.305

ASC_OUT = -1.67

In [37]:
kappa_records = []
inv_kappa_records = []
omega_records = []
g_records = []

for n in range(len(df)):
    age10 = df.iloc[n][ "Age"] / 10
    female = df.iloc[n][ "Female"]

    g_records.append(
        (str(n), float(df.iloc[n][ "volume_gemerated_per_user_grams"]))
    )

    for i in range(1, 36):
        dist = df.iloc[n][ f"dist_{i}"]
        indoor = df.iloc[n][ f"indoor_{i}"]

        building = (
            df.iloc[n][ f"private_building_{i}"]
            + df.iloc[n][ f"public_building_{i}"])

        business = df.iloc[n][ f"industrial_business_center_{i}"]

        campus = (
        df.iloc[n][f"university_{i}"]
        + df.iloc[n][f"technical_institution_{i}"]
        + df.iloc[n][f"science_park_{i}"])

        V_machine = (
            B_DIST * dist
            + B_DIST_INDOOR * dist * indoor
            + B_INDOOR_AGE * indoor * age10
            + B_INDOOR_FEMALE * indoor * female
            + B_BUILDING * building
            + B_BUSINESS * business
            + B_CAMPUS * campus
        )
        V = V_machine - ASC_OUT
        V = np.clip(V, -50, 50)

        exp_V = float(np.exp(V))
        

        kappa_value = exp_V
        kappa_value = max(kappa_value, 1e-9)

        inv_kappa_value = float(1 / kappa_value)
        omega_value = exp_V / (1 + exp_V)

        kappa_records.append((str(n), str(i), kappa_value))
        inv_kappa_records.append((str(n), str(i), inv_kappa_value))
        omega_records.append((str(n), str(i), omega_value))

In [38]:
print("users:", len(users))
print("machines:", len(machines))
print("kappa records:", len(kappa_records))
print("inv kappa records:", len(inv_kappa_records))
print("omega records:", len(omega_records))
print("g records:", len(g_records))

users: 50
machines: 35
kappa records: 1750
inv kappa records: 1750
omega records: 1750
g records: 50


In [41]:
m = Container()

N = Set(m, "N", records=users)
I = Set(m, "I", records=machines)

kappa = Parameter(m, "kappa", domain=[N, I], records=kappa_records)
inv_kappa = Parameter(m, "inv_kappa", domain=[N, I], records=inv_kappa_records)
omega = Parameter(m, "omega", domain=[N, I], records=omega_records)
g = Parameter(m, "g", domain=N, records=g_records)

In [43]:
open_rvm = Variable(m, "open_rvm", domain=I, type="Binary")
prob = Variable(m, "prob", domain=[N, I], type="Positive")
prob_out = Variable(m, "prob_out", domain=N, type="Positive")

In [45]:
prob_sum = Equation(m, "prob_sum", domain=N)
open_link = Equation(m, "open_link", domain=[N, I])
logit_upper = Equation(m, "logit_upper", domain=[N, I])
logit_lower = Equation(m, "logit_lower", domain=[N, I])
machines_number = Equation(m, "machines_number")

K = 10

prob_sum[N] = prob_out[N] + Sum(I, prob[N, I]) == 1

open_link[N, I] = prob[N, I] <= omega[N, I] * open_rvm[I]

logit_upper[N, I] = prob[N, I] <= kappa[N, I] * prob_out[N]

logit_lower[N, I] = (
    prob_out[N]
    - inv_kappa[N, I] * prob[N, I]
    + open_rvm[I]
    <= 1
)

machines_number[...] = Sum(I, open_rvm[I]) <= K

In [55]:
budget = 840_000_000
v = 2.3  # COP per unit of plastic volume
fixed_cost_rvm = 24_000_000
sample_weight = 10_000

# Veränderbarer Budgetparameter
budget_limit = Parameter(
    m,
    "budget_limit",
    records=840_000_000
)

budget_constraint = Equation(m, "budget_constraint")

budget_constraint[...] = (
    Sum(I, fixed_cost_rvm * open_rvm[I])
    <= budget_limit
)

revenue = Sum(
    (N, I),
    sample_weight * g[N] * v * prob[N, I]
)

fixed_costs = Sum(
    I,
    fixed_cost_rvm * open_rvm[I]
)

objective = revenue - fixed_costs

rvm_model = Model(
    m,
    "rvm_mip_monetary",
    equations=m.getEquations(),
    problem=Problem.MIP,
    sense=Sense.MAX,
    objective=objective
)

In [57]:
import pandas as pd
import numpy as np

# Wegen einheitlicher Fixkosten sind nur diese Budgets relevant:
budget_values = [
    number_of_machines * fixed_cost_rvm
    for number_of_machines in range(K + 1)
]

results = []

for current_budget in budget_values:

    # Budgetparameter aktualisieren
    budget_limit.setRecords(current_budget)

    # Modell erneut lösen
    rvm_model.solve(solver="highs")

    # Geöffnete Maschinen auslesen
    selected_records = open_rvm.records[
        open_rvm.records["level"] > 0.5
    ]

    selected_machines = tuple(
        selected_records["I"].astype(str).tolist()
    )

    # Tatsächlich verwendetes Budget
    used_budget = (
        len(selected_machines) * fixed_cost_rvm
    )

    # Mittlere Opt-out-Wahrscheinlichkeit
    average_opt_out = prob_out.records["level"].mean()

    # Gesamtwahrscheinlichkeit, irgendeine RVM zu wählen
    # Für jeden Nutzer gilt:
    # sum_i prob[n,i] = 1 - prob_out[n]
    average_choice_probability = 1 - average_opt_out

    results.append({
        "Available budget": current_budget,
        "Budget capacity": current_budget // fixed_cost_rvm,
        "Used budget": used_budget,
        "Objective": rvm_model.objective_value,
        "Revenue": revenue.toValue(),
        "Fixed costs": fixed_costs.toValue(),
        "Number selected": len(selected_machines),
        "Selected machines": selected_machines,
        "Average choice probability": average_choice_probability,
        "Average opt-out probability": average_opt_out,
        "Solver time": rvm_model.total_solver_time
    })

breakpoint_df = pd.DataFrame(results)

breakpoint_df

,Available budget,Budget capacity,Used budget,Objective,Revenue,Fixed costs,Number selected,Selected machines,Average choice probability,Average opt-out probability,Solver time
0,0,0,0,0.000000e+00,0.000000e+00,0.000000e+00,0,(),0.000000,1.000000,0.086
1,24000000,1,24000000,2.048303e+08,2.288303e+08,2.400000e+07,1,"(4,)",0.941264,0.058736,0.236
2,48000000,2,24000000,2.048303e+08,2.288303e+08,2.400000e+07,1,"(4,)",0.941264,0.058736,0.966
3,72000000,3,24000000,2.048303e+08,2.288303e+08,2.400000e+07,1,"(4,)",0.941264,0.058736,1.000
4,96000000,4,24000000,2.048303e+08,2.288303e+08,2.400000e+07,1,"(4,)",0.941264,0.058736,0.986
5,120000000,5,24000000,2.048303e+08,2.288303e+08,2.400000e+07,1,"(4,)",0.941264,0.058736,0.936
6,144000000,6,24000000,2.048303e+08,2.288303e+08,2.400000e+07,1,"(4,)",0.941264,0.058736,0.930
7,168000000,7,24000000,2.048303e+08,2.288303e+08,2.400000e+07,1,"(4,)",0.941264,0.058736,0.927
8,192000000,8,24000000,2.048303e+08,2.288303e+08,2.400000e+07,1,"(4,)",0.941264,0.058736,0.933
9,216000000,9,24000000,2.048303e+08,2.288303e+08,2.400000e+07,1,"(4,)",0.941264,0.058736,0.920


In [59]:
rvm_model.solve(solver="highs")

print("Objective value (COP):", rvm_model.objective_value)
print("Revenue (COP):", revenue.toValue())
print("Fixed costs (COP):", fixed_costs.toValue())

print("\nSelected machines:")
print(open_rvm.records[open_rvm.records.level > 0.5])

Objective value (COP): 204830253.47304183
Revenue (COP): 228830253.47304094
Fixed costs (COP): 23999999.999999106

Selected machines:
   I  level  marginal  lower  upper  scale
3  4    1.0       NaN    0.0    1.0    1.0


In [51]:
print("Max possible revenue if all plastic captured:", sum(df["volume_gemerated_per_user_grams"]) * 2.3)
print("Cost of one RVM:", fixed_cost_rvm)

Max possible revenue if all plastic captured: 24298.465000782
Cost of one RVM: 24000000


In [53]:
print("Selected machines:")
print(open_rvm.records[open_rvm.records["level"]==1])

print("Choice probabilities:")
print(prob.records.head(50))

print("Opt_Out probabilities:")
print(prob_out.records)

Selected machines:
   I  level  marginal  lower  upper  scale
3  4    1.0       NaN    0.0    1.0    1.0
Choice probabilities:
    N   I     level  marginal  lower  upper  scale
0   0   1  0.000000       NaN    0.0    inf    1.0
1   0   2  0.000000       NaN    0.0    inf    1.0
2   0   3  0.000000       NaN    0.0    inf    1.0
3   0   4  0.926638       NaN    0.0    inf    1.0
4   0   5  0.000000       NaN    0.0    inf    1.0
5   0   6  0.000000       NaN    0.0    inf    1.0
6   0   7  0.000000       NaN    0.0    inf    1.0
7   0   8  0.000000       NaN    0.0    inf    1.0
8   0   9  0.000000       NaN    0.0    inf    1.0
9   0  10  0.000000       NaN    0.0    inf    1.0
10  0  11  0.000000       NaN    0.0    inf    1.0
11  0  12  0.000000       NaN    0.0    inf    1.0
12  0  13  0.000000       NaN    0.0    inf    1.0
13  0  14  0.000000       NaN    0.0    inf    1.0
14  0  15  0.000000       NaN    0.0    inf    1.0
15  0  16  0.000000       NaN    0.0    inf    1.0
16  0 

In [27]:
import sys

In [29]:
print(f"Solver runtime: {rvm_model.total_solver_time:.4f} seconds")

Solver runtime: 1.4930 seconds
